In [1]:
import torch
import matplotlib.pyplot as plt

from src.tokenizer import CharTokenizer
from src.model import GPTModel
from src.utils import generate_causal_mask
from data.prepare_data import load_data

from configs.config import *

ModuleNotFoundError: No module named 'torch'

In [ ]:
text = load_data("../data/input.txt")

print("Dataset length:", len(text))
print(text[:200])  # preview

In [ ]:
tokenizer = CharTokenizer(text)

sample = "hello world"
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decoded)
print("Vocab size:", tokenizer.vocab_size)

In [ ]:
x = torch.randint(0, tokenizer.vocab_size, (4, 10))  # (batch=4, seq=10)

print("Input shape:", x.shape)

In [ ]:
model = GPTModel(
    tokenizer.vocab_size,
    embed_dim,
    num_heads,
    num_layers,
    ff_dim,
    max_len
).to(device)

x = x.to(device)
mask = generate_causal_mask(x.size(1)).to(device)

out = model(x, mask)

print("Output shape:", out.shape)

In [ ]:
logits = out[0, -1]  # last token prediction
probs = torch.softmax(logits, dim=-1)

top_token = torch.argmax(probs).item()
print("Predicted token:", top_token)
print("Character:", tokenizer.decode([top_token]))

In [ ]:
model.eval()

start = "To be"
tokens = tokenizer.encode(start)

for _ in range(50):
    x = torch.tensor(tokens).unsqueeze(0).to(device)
    mask = generate_causal_mask(x.size(1)).to(device)

    logits = model(x, mask)
    next_token = torch.argmax(logits[0, -1]).item()

    tokens.append(next_token)

print(tokenizer.decode(tokens))

In [ ]:
# simulate loss values
losses = [5.0, 4.2, 3.8, 3.2, 2.9]

plt.plot(losses)
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
# Just check shapes inside attention
for name, param in model.named_parameters():
    print(name, param.shape)